# 🗄️ Notebook 6: Vector DB Setup (Medical Books)
**Goal:** Chunk the Medical Textbooks (PDFs) and embed them into ChromaDB for semantic search.

**Language Note:** The PDFs might be in English, but since we are using `multilingual-e5-large`, the model places English and Gujarati in the same semantic space. A Gujarati query can retrieve the exact English medical text!

**Input:** `data/books/*.pdf`
**Output:** ChromaDB directory at `data/chroma_db_books/`

In [ ]:
!pip install -q chromadb sentence-transformers langchain langchain-community PyMuPDF
import os
import glob
import fitz  # PyMuPDF
import chromadb
from langchain.text_splitter import RecursiveCharacterTextSplitter

os.makedirs("data/chroma_db_books", exist_ok=True)
print("✅ Libraries loaded")

## 1. Extract Text from PDFs

In [ ]:
pdf_files = glob.glob("data/books/*.pdf")
raw_docs = []

if not pdf_files:
    print("⚠️ No PDF books found in 'data/books/'.")
    print(
        "Using a sample fallback text for demonstration so the Vector DB builds successfully."
    )
    sample_text = """Diabetes mellitus causes symptoms like frequent urination, excessive thirst, and fatigue. 
    Treatment for diabetes includes Metformin, insulin therapy, and dietary changes.
    Hypertension (high blood pressure) symptoms include headaches and shortness of breath, treated with Amlodipine.
    Malaria causes severe fever and chills; it is treated with Chloroquine or Artemisinin.
    ડેન્ગ્યુ (Dengue) ના લક્ષણો તાવ (fever) અને સાંધાનો દુખાવો છે, Paracetamol આપી શકાય.
    """
    raw_docs.append({"text": sample_text, "source": "sample_medical_data"})
else:
    print(f"Found {len(pdf_files)} PDF(s). Extracting text...")
    for file_path in pdf_files:
        doc_name = os.path.basename(file_path)
        print(f"Reading: {doc_name}")
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            text += page.get_text() + "\n"
        raw_docs.append({"text": text, "source": doc_name})

print(f"✅ Extracted text from {len(raw_docs)} document(s).")

## 2. Text Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunked_texts = []
chunked_metadatas = []
chunked_ids = []

for doc_idx, doc in enumerate(raw_docs):
    splits = text_splitter.split_text(doc["text"])
    for split_idx, split in enumerate(splits):
        # Clean up excessive whitespace
        clean_split = " ".join(split.split())
        if len(clean_split) < 20:
            continue  # Skip tiny fragments

        chunked_texts.append(clean_split)
        chunked_metadatas.append({"source": doc["source"], "chunk": split_idx})
        chunked_ids.append(f"doc{doc_idx}_chunk{split_idx}")

print(f"✅ Split PDFs into {len(chunked_texts)} text chunks.")
print(f"Example chunk: {chunked_texts[0][:150]}...")

## 3. Initialize Embedding Model & ChromaDB

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading multilingual embedding model on {device}...")

# e5 requires "query: " or "passage: " prefixes
embed_model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)

client = chromadb.PersistentClient(path="data/chroma_db_books")
try:
    client.delete_collection(name="medical_books")
except Exception:
    pass

collection = client.create_collection(
    name="medical_books", metadata={"hnsw:space": "cosine"}
)
print("✅ Embedding model and ChromaDB initialized.")

## 4. Compute Embeddings and Store

In [ ]:
# Add passage prefix for e5
passages_to_embed = [f"passage: {t}" for t in chunked_texts]

print("Computing embeddings... this may take a moment depending on the PDF sizes.")
batch_size = 64
for i in range(0, len(passages_to_embed), batch_size):
    text_batch = chunked_texts[i : i + batch_size]
    embed_batch = passages_to_embed[i : i + batch_size]
    meta_batch = chunked_metadatas[i : i + batch_size]
    id_batch = chunked_ids[i : i + batch_size]

    vectors = embed_model.encode(embed_batch, convert_to_numpy=True).tolist()

    collection.add(
        embeddings=vectors, documents=text_batch, metadatas=meta_batch, ids=id_batch
    )

print(
    f"✅ Successfully indexed {collection.count()} book chunks into ChromaDB at `data/chroma_db_books/`"
)

## 5. Test Retrieval (Cross-Lingual)

In [ ]:
test_query = "ડાયાબિટીઝ ના મુખ્ય લક્ષણો શું છે?"  # Gujarati query for "What are main diabetes symptoms?"
print(f"Testing cross-lingual retrieval for query: '{test_query}'\n")

query_vec = embed_model.encode(f"query: {test_query}", convert_to_numpy=True).tolist()

results = collection.query(query_embeddings=[query_vec], n_results=2)

for i, (doc, dist) in enumerate(zip(results["documents"][0], results["distances"][0])):
    print(f"[Result {i + 1}] (Distance: {dist:.4f})")
    print(f"{doc}\n")

print("✅ Vector DB is ready and working.")
print("\n📝 NEXT STEP: Run Notebook 07 to update the hybrid retrieval module.")